# 01 · Caso de negocio — RiskVision

Antes de abrir un solo dato, el proyecto necesita **una pregunta de negocio, una métrica que la refleje y un costo de equivocarse**.
Este notebook termina con esas tres cosas escritas **con números**, y su resumen va a `reports/00_caso.md`.

**Grupo 6 (Analistas de Datos):** Juan David Ardila Arboleda, Lina Isabel Vasquez Villegas, Andres Felipe Orozco Roa, Cristina Garces y Estefania Cervera.
**Dataset:** Default of Credit Card Clients (UCI Machine Learning Repository), clientes de un banco de Taiwán.


## 1. La pregunta de negocio

**Pregunta de negocio:** ¿Qué clientes de tarjeta de crédito tienen alta probabilidad de no pagar el próximo mes?

**¿Quién decide y con qué información?** El área de riesgo de crédito del banco. Cuenta con el límite de crédito, datos demográficos, el estado de pago de los últimos 6 meses y los montos facturados y pagados. Con la probabilidad de impago de cada cliente decide si ajusta el límite, hace seguimiento preventivo u ofrece reestructurar la deuda.

**¿Qué pasa hoy, sin el modelo?** Sin modelo no hay un orden de prioridad por riesgo: el equipo revisa a los clientes con reglas simples o sin criterio uniforme, y no sabe a quién contactar primero. *(Descripción general del caso: los datos no dicen cómo trabaja hoy el banco; el grupo debe confirmarlo.)*


## 2. Equivocarse no cuesta lo mismo

**Error más caro:** no detectar a un cliente que sí va a incumplir (falso negativo). El banco pierde el dinero prestado y no actúa a tiempo.

**Error menos caro:** marcar como riesgoso a un cliente que sí iba a pagar (falso positivo). Solo cuesta un seguimiento innecesario o un límite más bajo para un buen cliente.

**Proporción entre los dos (por ejemplo, 5 a 1) y de dónde sale:** **5 a 1. Es un supuesto del grupo, no un dato del dataset**: la base no trae costos reales (el notebook de selección de modelo lo indica). Se usa 5 a 1 como valor de trabajo y debe validarse con el instructor o con datos del banco. La matriz se puede cambiar abajo y todo lo demás se recalcula.


In [1]:
import numpy as np
import pandas as pd

CLASSES = ["paga", "incumple"]

# Filas = lo que realmente pasó, columnas = lo que predijo el modelo. Diagonal = 0.
# SUPUESTO: no detectar un "incumple" cuesta 5 veces más que una falsa alarma.
COST_MATRIX = [[0, 1], [5, 0]]

cost = pd.DataFrame(COST_MATRIX, index=[f"real {c}" for c in CLASSES], columns=[f"predice {c}" for c in CLASSES])
display(cost)

,predice paga,predice incumple
real paga,0,1
real incumple,5,0


## 3. ¿Contra qué nos comparamos? (línea base)

Una métrica sola no dice nada: necesita un punto de referencia. Calcula el costo de **dos estrategias sin modelo**:
- **Siempre la clase más frecuente** (o, en regresión, siempre el promedio).
- **Azar según las proporciones.**

Las proporciones reales vienen del conjunto de datos completo (30.000 clientes): 77,88 % paga y 22,12 % incumple.


In [2]:
# Proporciones en el mismo orden que CLASSES: 23.364 pagan y 6.636 incumplen, de 30.000 clientes.
priors = np.array([23364, 6636]) / 30000

cm = np.array(COST_MATRIX, dtype=float)
most_frequent = int(priors.argmax())
always_most_frequent = priors @ cm[:, most_frequent]
random_by_priors = priors @ cm @ priors
always_flag = priors @ cm[:, 1]
print(f"Siempre '{CLASSES[most_frequent]}':  costo esperado = {always_most_frequent:.2f}")
print(f"Azar según proporciones:   costo esperado = {random_by_priors:.2f}")
print(f"(Referencia) Marcar a todos como 'incumple': costo esperado = {always_flag:.2f}")

Siempre 'paga':  costo esperado = 1.11
Azar según proporciones:   costo esperado = 1.03
(Referencia) Marcar a todos como 'incumple': costo esperado = 0.78


**Lo que dice la línea base (una frase):** Un modelo que siempre dice «paga» acierta 77,9 % de las veces pero no detecta a ningún cliente que incumple, y con el costo 5 a 1 su costo esperado (1,11) es peor que el del azar (1,03), así que cualquier modelo útil debe bajar de ahí. Ojo: con 5 a 1, marcar a **todos** como «incumple» cuesta aún menos (0,78), porque casi no tiene falsos negativos. Ese es el verdadero piso a superar, y por eso el costo de revisar a cada cliente marcado (que hoy no tenemos) es el dato clave para fijar el umbral.

**¿La exactitud (accuracy) sirve como métrica en tu caso? ¿Por qué sí o por qué no?** No. Con 77,9 % de clientes que pagan, el modelo trivial ya tiene 77,9 % de exactitud sin detectar un solo incumplimiento, y la exactitud trata igual los dos errores aunque no cuesten lo mismo.


## 4. Cómo se medirá el éxito

- **Métrica principal:** ROC AUC — mide qué tan bien el modelo ordena a los clientes de menor a mayor riesgo sin fijar un umbral, algo adecuado con clases desbalanceadas y sin costos reales en los datos. 0,5 es azar y 1,0 es separación perfecta.
- **Métricas de control:** recall de la clase «incumple» (por el costo del falso negativo), precision, F1, AUC-PR y el costo esperado con la matriz de arriba una vez elegido un umbral.
- **Qué resultado haría que el negocio diga «sirve»:** un ROC AUC en prueba claramente por encima de 0,5000 (línea base). *Referencia del grupo:* el notebook de selección de modelo obtuvo **0,7784** en prueba con Gradient Boosting, y esa cifra es el punto de partida a mantener o mejorar. La meta exacta debe fijarla el grupo con el instructor.


## 5. Lo que podría salir mal

- **Fuga de información (¿qué columna se conoce solo después de la decisión?):** Las columnas `PAY_*`, `BILL_AMT*` y `PAY_AMT*` corresponden a abril–septiembre, antes del mes que se predice, así que no se ve una fuga evidente. El riesgo real está en el proceso: escalar o imputar antes de partir los datos, o usar la prueba para elegir modelo o umbral. Todo se ajusta solo con entrenamiento.
- **Sesgos o equidad (¿a quién podría perjudicar el modelo?):** `SEX`, `AGE` y `MARRIAGE` son atributos sensibles; el modelo podría perjudicar a un grupo. Se compararán los errores por grupo en la etapa de validación.
- **Datos personales o confidenciales:** No hay nombres ni documentos, solo un `ID` anónimo, que se excluye del modelo. La licencia del dataset se debe verificar en la página de UCI.
- **Hasta dónde puede decidir el modelo y dónde debe decidir una persona:** El modelo ordena por riesgo y prioriza revisiones; no aprueba ni niega crédito. La decisión final sobre un cliente la toma una persona del área de riesgo. Además, los datos son de un solo banco de Taiwán en 2005 y pueden no servir en otro país o época.


## 6. Cierre

- [x] Pregunta, quién decide y costos escritos **con números** (la proporción 5 a 1 es un supuesto por validar).
- [x] Línea base calculada.
- [x] Métrica principal elegida y justificada.
- [x] Resumen copiado a `reports/00_caso.md`.
- [ ] Alguien que **no** escribió este notebook sabe explicar por qué un error cuesta más que el otro.
